# Notebook assumptions

* anaconda and postgres are running via docker-compose containers

# imports / helper methods / DB connections

In [13]:
import csv

import math
import numpy as np
import pandas as pd

import psycopg2

In [14]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)
cursor = connection.cursor()

In [15]:
def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

# create temp tables

In [27]:
connection.rollback()
cursor.execute("""

drop table if exists movies;

""")
connection.rollback()

In [21]:
connection.rollback()
cursor.execute("""
    
    create table movies (

        id integer primary key,             --  0
        "index" integer,                    --  1
        budget integer,                     --  2
        genres varchar(255),                --  3
        homepage varchar(255),              --  4
        keywords varchar(255),              --  5
        original_language varchar(255),     --  6
        original_title varchar(255),        --  7
        overview varchar(255),              --  8
        popularity decimal,                 --  9
        production_companies varchar(255),  -- 10
        production_countries varchar(255),  -- 11
        release_date date,                  -- 12
        revenue integer,                    -- 13
        runtime decimal,                    -- 14
        spoken_languages varchar(255),      -- 15
        status varchar(255),                -- 16
        tagline varchar(255),               -- 17
        title varchar(255),                 -- 18
        vote_average decimal,               -- 19
        vote_count integer,                 -- 20
        "cast" varchar(255),                -- 21
        crew varchar(255),                  -- 22
        director varchar(255)               -- 23
    );
    
    """)
connection.rollback()

In [31]:

movies_csv = "../data/raw/movies.csv"

movies_file = open(movies_csv, "r")
movies_reader = csv.reader(movies_file)

i = 0
print_limit = 10

for movie in movies_reader:
    i += 1
    if i > 1 and i <= print_limit + 1:
        print(f"[{movie[0]}] {movie[18]} ({movie[23]})")

print("\nPrinted ", print_limit, "movies (of ", i, "total).")

[0] Avatar (James Cameron)
[1] Pirates of the Caribbean: At World's End (Gore Verbinski)
[2] Spectre (Sam Mendes)
[3] The Dark Knight Rises (Christopher Nolan)
[4] John Carter (Andrew Stanton)
[5] Spider-Man 3 (Sam Raimi)
[6] Tangled (Byron Howard)
[7] Avengers: Age of Ultron (Joss Whedon)
[8] Harry Potter and the Half-Blood Prince (David Yates)
[9] Batman v Superman: Dawn of Justice (Zack Snyder)

Printed  10 movies (of  4804 total).


In [ ]:
""